In [1]:
import pandas as pd
import geopandas as gpd
from pyprojroot import here

In [2]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [3]:
mun = gpd.read_file(RAW_DATA_DIR/"geography/municipalities/00mun.shp")
highways = gpd.read_file(RAW_DATA_DIR/"logistics/highways_gpkg/rnc2025.gpkg", layer="red_vial")

In [4]:
# We keep only highways (CARRETERA)
keep     = ["CARRETERA"]
filtered_highways      = highways[
    (highways["TIPO_VIAL"].str.upper().str.strip().isin([t.upper() for t in keep]))
].copy()

filtered_highways = filtered_highways.to_crs(mun.crs)
print(f"Highways found: {len(filtered_highways)}")

Highways found: 333968


In [5]:
join = gpd.sjoin(
    mun[["CVEGEO", "geometry"]], 
    filtered_highways[["geometry"]], 
    how="left", 
    predicate="intersects"
)

In [6]:
mun["highways"] = mun.index.map(
    join.groupby(join.index)["index_right"].apply(lambda x: int(x.notna().any()))
)

In [8]:
result = mun[["CVEGEO", "highways"]]
result.to_csv(INTERIM_DATA_DIR/"highways_by_municipality.csv", index=False)

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
     CVEGEO  highways
580   14055         1
1834  24029         1
249   08013         1
452   12026         1
1720  21159         1
1242  20492         1
1256  20007         0
644   14096         1
2172  30190         1
671   15072         1
